# Knowledge and Data: Practical Assignment 3 
## RDF Data, RDFS knowledge and inferencing 

YOUR NAME: Boris Fortuin    

YOUR VUNetID: DBO737

*(If you do not provide your name and VUNetID we will not accept your submission).* 

### Learning objectives

At the end of this exercise you should be able to:

1. Access local an external data via SPARQL both from within a python programming environment and stand-alone with a GUI, such as [YASGUI](https://yasgui.triply.cc/), and this way integrate data from different sources  
2. Model your own first knowledge base, in this case an RDF Schema knowledge graph
3. Implement inference rules 

Follow this Notebook step-by-step. 

Of course, you can do the exercises in any Programming Editor of your liking. 
But you do not have to. Feel free to simply write code in the Notebook. When 
everythink is filled in and works, safe the Notebook and submit it 
as a Jupyter Notebook, i.e. with an ipynb extension. Please use as name of the 
Notebook your studentID+Assignment3.ipynb.  


We will not evaluate the programming style of your solutions. Yet we do look whether your solutions suggests an understanding, and whether they yield the correct output.

Note that all notebooks will automatically be checked for plagiarism: while similar answers can be expected, it is not allowed to directly copy the solutions from fellow students or TAs, or from the examples discussed during the lectures. Similarly, sharing your solutions with your peers is not allowed.

**IMPORTANT: Submit this notebook after finishing the assignment. It is not necessary to submit the created turtle files**

Before you start, you need to:

- **Install the *rdflib* Python package:** *pip install rdflib* (should already be installed from the previous assignment)
- **Install the *SPARQLWrapper* Python package:** *pip install SPARQLWrapper*
- **Install the free edition of the GraphDB Triplestore:** please follow this short [GraphDB tutorial](https://github.com/ucds-vu/knowledge-data-vu/blob/master/Tutorials/Preliminaries/tutorial-GraphDB.md). 

Then, add the file example-from-slides.ttl to a newly created database, say called assignment-3. 

**Note that you should have an active internet connection to run the code in this notebook. If, for some external reason (ie internet and/or system issues), you cannot access the SPARQL endpoint, then report this to a TA as soon as possible!**

In [1]:
# install library
%pip install SPARQLWrapper

Note: you may need to restart the kernel to use updated packages.


c:\Users\Boris\OneDrive - Vrije Universiteit Amsterdam\Everything for VU\year 3\minor\Knowledge and Data\knowledge-data-vu\.venv\Scripts\python.exe: No module named pip


## Task 1: (35 points) Integrate Local and External Data

You can integrate SPARQL queries into your Python code by using the *RDFLib* and *SPARQLWrapper* libraries. 

The following code accesses the DBPedia knowledge graph using its SPARQL endpoint, and returns the result of the SPARQL query requesting all the labels asserted to Amsterdam (test it!)  

In [11]:
# This code only works if you are online.
# If, for some reason, you cannot get this to work, then please contact a TA

from rdflib import Graph, RDF, RDFS, Namespace, Literal, URIRef
from SPARQLWrapper import SPARQLWrapper, JSON

sparql = SPARQLWrapper("http://dbpedia.org/sparql")
sparql.setQuery("""
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    SELECT ?cityName
    WHERE { 
        <http://dbpedia.org/resource/Amsterdam> rdfs:label ?cityName 
    }
""")
sparql.setReturnFormat(JSON)
results = sparql.query().convert()
for result in results["results"]["bindings"]:
    print(result["cityName"]["value"])  

Amsterdam
Amsterdam
Amsterdam
Ámsterdam
أمستردام
Amsterdam
Amsterdam
Άμστερνταμ
Amsterdamo
Amsterdam
Amstardam
Amsterdam
Amsterdam
Amsterdam
アムステルダム
암스테르담
Amsterdam
Amesterdão
Амстердам
Amsterdam
Амстердам
阿姆斯特丹


For your convenience, we already wrote the following functions that might be useful to complete this task. 
In addition, we have loaded and printed the 'example-from-slides.ttl' dataset.

In [12]:
from rdflib import Graph, RDF, Namespace, Literal, URIRef
from SPARQLWrapper import SPARQLWrapper, JSON


# Loads the data from a certain file given as input in Turtle syntax into the Graph g  
# -------------------------
def load_graph(graph, filename):
    with open(filename, 'r') as f:
        graph.parse(f, format='turtle')
        

# Prints a certain graph given as input in Turtle syntax
# if your output shows byte string (ie, b'...') you must add '.decode()' to the print statements:
#    print(myGraph.serialize(format='turtle').decode())
# -------------------------
def serialize_graph(myGraph):
     print(myGraph.serialize(format='turtle'))
        

# Saves the Graph g in Turtle syntax to a certain file given as input
# -------------------------
def save_graph(myGraph, filename):
    with open(filename, 'w') as f:
        myGraph.serialize(filename, format='turtle')
        
    
# Changes the namespace of a certain URI given as input to a DBpedia URI 
# Example: transformToDBR("http://example.com/kad2020/Amsterdam") returns "http://dbpedia.org/resource/Amsterdam"
# -------------------------
def transformToDBR(uri):
    if isinstance(uri, Literal):
        # changes the literal to uppercase so that the object with the same name refers to an object and not the string
        return uri.upper()
    components = g.namespace_manager.compute_qname(uri)
    return "http://dbpedia.org/resource/%s"%(components[2])

# -------------------------

g = Graph()
load_graph(g, 'example-from-slides.ttl')
serialize_graph(g)


# Don't forget to run this cell before continuing the task.


@prefix ex: <http://example.com/kad/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

ex:Bucharest a ex:Capital .

ex:Netherlands a ex:Country ;
    ex:contains ex:Ijsselmeer ;
    ex:containsCity ex:Rotterdam ;
    ex:hasCapital ex:Amsterdam ;
    ex:hasName "The Netherlands" ;
    ex:neighbours ex:Belgium .

ex:Oradea a ex:Capital .

ex:hasCapital rdfs:range ex:Capital ;
    rdfs:subPropertyOf ex:containsCity .

ex:neighbours rdfs:subPropertyOf ex:closeBy .

ex:Amsterdam a ex:Capital ;
    ex:closeBy ex:Germany .

ex:Belgium a ex:Country .

ex:EuropeanCountry rdfs:subClassOf ex:Country .

ex:Germany a ex:EuropeanCountry ;
    ex:hasCapital ex:Berlin .

ex:closeBy rdfs:domain ex:Location ;
    rdfs:range ex:Location .

ex:containsCity rdfs:domain ex:Country ;
    rdfs:range ex:City ;
    rdfs:subPropertyOf ex:contains .

ex:City rdfs:subClassOf ex:Location .

ex:Capital rdfs:subClassOf ex:City .

ex:Country rdfs:subClassOf ex:Location .




### A: Write a SPARQL query that finds all the cities in the dataset

As you cannot directly use class City, you will have to find those cities in the dataset (example-from-slides.ttl) using implicit information that can be deduced from the domain and ranges of the relations (e.g. things in a hasCapital relation are capitals and a capital is a city, etc.).

Save all the cities returned from the SPARQL query into the empty set "cities". 

In [13]:
cities = set()

query = """
PREFIX ex: <http://example.com/kad/>

SELECT ?city
WHERE {
  {
    ?city a ex:City .
  }
  UNION
  {
    ?city a ex:Capital .
  }
  UNION
  {
    ?country ex:hasCapital ?city .
  }
  UNION
  {
    ?country ex:containsCity ?city .
  }
}
"""

for row in g.query(query):
    cities.add(str(row.city))

for city in sorted(cities):
    print(city)

http://example.com/kad/Amsterdam
http://example.com/kad/Berlin
http://example.com/kad/Bucharest
http://example.com/kad/Oradea
http://example.com/kad/Rotterdam


### B: For each city, find from DBpedia its longitude & latitude, and its number of inhabitants (if available)

Don't forget to adapt the namespace of the cities in your dataset when querying DBpedia, using the above function *transformToDBR(uri)*. Also note that namespaces should never use the *https* protocol.

The empty graph h should only contain the triples extracted from DBpedia, but added to the URIs with the 'ex' namespace. 
An example of a triple in h is the following triple: 
       
       ex:Amsterdam dbo:populationTotal "872680"^^xsd:nonNegativeInteger .

In [ ]:
h = Graph()

for city in cities:
    #build the URI's
    city_uri = URIRef(city)
    dbpedia_uri = URIRef(transformToDBR(city_uri))
    ex_city = URIRef("http://example.com/kad/" + city_uri.split('/')[-1])

    #query the database for information. I use OPTIONAL for incase any infromation i lacking
    sparql = SPARQLWrapper("http://dbpedia.org/sparql")
    sparql.setQuery(f"""
        PREFIX dbo: <http://dbpedia.org/ontology/>
        PREFIX geo: <http://www.w3.org/2003/01/geo/wgs84_pos#>

        SELECT ?lat ?long ?pop
        WHERE {{
            <{dbpedia_uri}> geo:lat ?lat ;
                            geo:long ?long .
            OPTIONAL {{ <{dbpedia_uri}> dbo:populationTotal ?pop . }}
        }}
    """)
    sparql.setReturnFormat(JSON)
    results = sparql.query().convert()

    #adding the subject, predicate and object as a triple to graph h
    if(results["results"]["bindings"]):
        row = results["results"]["bindings"][0]
        if("lat" in row):
            h.add((ex_city, URIRef("http://www.w3.org/2003/01/geo/wgs84_pos#lat"), Literal(row["lat"]["value"])))
        if("long" in row):
            h.add((ex_city, URIRef("http://www.w3.org/2003/01/geo/wgs84_pos#long"), Literal(row["long"]["value"])))
        if("pop" in row):
            h.add((ex_city, URIRef("http://dbpedia.org/ontology/populationTotal"), Literal(row["pop"]["value"])))

serialize_graph(h)

@prefix ns1: <http://www.w3.org/2003/01/geo/wgs84_pos#> .
@prefix ns2: <http://dbpedia.org/ontology/> .

<http://example.com/kad/Amsterdam> ns2:populationTotal "933680" ;
    ns1:lat "52.37277603149414" ;
    ns1:long "4.893610954284668" .

<http://example.com/kad/Berlin> ns2:populationTotal "3596999" ;
    ns1:lat "52.52000045776367" ;
    ns1:long "13.40499973297119" .

<http://example.com/kad/Bucharest> ns2:populationTotal "1877155" ;
    ns1:lat "44.43249893188477" ;
    ns1:long "26.10388946533203" .

<http://example.com/kad/Oradea> ns2:populationTotal "183105" ;
    ns1:lat "47.07222366333008" ;
    ns1:long "21.92111206054688" .

<http://example.com/kad/Rotterdam> ns2:populationTotal "868135" ;
    ns1:lat "51.91999816894531" ;
    ns1:long "4.480000019073486" .




### C: Save your results

- Merge the triples from example-from-slides.ttl with the information extracted from DBpedia. See the [documentation](https://rdflib.readthedocs.io/en/stable/merging.html) on how to accomplish this.
- Save all these triples into a new file 'extended-example.ttl'. **It is not necessary to submit this file**
- Print all triples in Turtle Syntax.


In [18]:
combined = Graph()
combined += g
combined += h

save_graph(combined, 'extended-example.ttl')
serialize_graph(combined)

@prefix ns1: <http://www.w3.org/2003/01/geo/wgs84_pos#> .
@prefix ns2: <http://example.com/kad/> .
@prefix ns3: <http://dbpedia.org/ontology/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

ns2:Bucharest a ns2:Capital ;
    ns3:populationTotal "1877155" ;
    ns1:lat "44.43249893188477" ;
    ns1:long "26.10388946533203" .

ns2:Netherlands a ns2:Country ;
    ns2:contains ns2:Ijsselmeer ;
    ns2:containsCity ns2:Rotterdam ;
    ns2:hasCapital ns2:Amsterdam ;
    ns2:hasName "The Netherlands" ;
    ns2:neighbours ns2:Belgium .

ns2:Oradea a ns2:Capital ;
    ns3:populationTotal "183105" ;
    ns1:lat "47.07222366333008" ;
    ns1:long "21.92111206054688" .

ns2:hasCapital rdfs:range ns2:Capital ;
    rdfs:subPropertyOf ns2:containsCity .

ns2:neighbours rdfs:subPropertyOf ns2:closeBy .

ns2:Amsterdam a ns2:Capital ;
    ns3:populationTotal "933680" ;
    ns2:closeBy ns2:Germany ;
    ns1:lat "52.37277603149414" ;
    ns1:long "4.893610954284668" .

ns2:Belgium a ns2:Country

## Task 2: (25 points)  Implement Basic Inferencing Rules 

In the lecture we showed that the RDFS inference rules can be used to infer new knowledge. For example, infer class membership based on _rdfs:domain_ or infer relationships between subjects and objects based on _rdfs:subPropertyOf_. 

Create rules to inference class membership based on the RDF Schema language features 
*	For example: infer that an instance belongs to a class because of domain and range restrictions
*	For example: infer that an instance belongs to a (super)class because it also belongs to a subclass

We implemented the __rdfs2__ rule. You should implement the 5 following remaining rules:  

*     (rdfs2) If G contains the triples (aaa rdfs:domain xxx.) and (uuu aaa yyy.)  then infer the triple (uuu rdf:type xxx.)
*     (rdfs3) If G contains the triples (aaa rdfs:range xxx.) and (uuu aaa vvv.) then infer the triple (vvv rdf:type xxx .)
*     (rdfs5) If G contains the triples (uuu rdfs:subPropertyOf vvv.) and (vvv rdfs:subPropertyOf xxx.) then infer the triple (uuu rdfs:subPropertyOf xxx .) 
*     (rdfs7) If G contains the triples (aaa rdfs:subPropertyOf bbb.) and (uuu aaa yyy.) then infer the triple (uuu bbb yyy .) 
*     (rdfs9) If G contains the triples (uuu rdfs:subClassOf xxx.) and (vvv rdf:type uuu.) then infer the triple (vvv rdf:type xxx .) 
*     (rdfs11) If G contains the triples (uuu rdfs:subClassOf vvv.) and (vvv rdfs:subClassOf xxx.) then infer the triple (uuu rdfs:subClassOf xxx .)


Run your rule reasoner on your knowledge graph. If you have implemented everything correctly, you should find exactly 17 inferences.

In [ ]:
def myRDFSreasoner(myGraph):
    inferredTriples = set()

    # rdfs2: (aaa rdfs:domain xxx.) and (uuu aaa yyy.) => (uuu rdf:type xxx.)
    for prop, domain in myGraph.subject_objects(RDFS.domain):
        for subject, value in myGraph.subject_objects(prop):
            inferredTriples.add((subject, RDF.type, domain))
            print("(rdfs2)", subject, "rdf:type", domain)

    # rdfs3: (aaa rdfs:range xxx.) and (uuu aaa vvv.) => (vvv rdf:type xxx.)
    for prop, rng in myGraph.subject_objects(RDFS.range):
        for subject, value in myGraph.subject_objects(prop):
            inferredTriples.add((value, RDF.type, rng))
            print("(rdfs3)", value, "rdf:type", rng)

    # rdfs5: (uuu rdfs:subPropertyOf vvv.) and (vvv rdfs:subPropertyOf xxx.)
    #        => (uuu rdfs:subPropertyOf xxx.)
    for prop1, superprop1 in myGraph.subject_objects(RDFS.subPropertyOf):
        for prop2, superprop2 in myGraph.subject_objects(superprop1):
            inferredTriples.add((prop1, RDFS.subPropertyOf, superprop2))
            print("(rdfs5)", prop1, "rdfs:subPropertyOf", superprop2)

    # rdfs7: (aaa rdfs:subPropertyOf bbb.) and (uuu aaa yyy.) => (uuu bbb yyy.)
    for prop, superprop in myGraph.subject_objects(RDFS.subPropertyOf):
        for subject, value in myGraph.subject_objects(prop):
            inferredTriples.add((subject, superprop, value))
            print("(rdfs7)", subject, superprop, value)

    # rdfs9: (uuu rdfs:subClassOf xxx.) and (vvv rdf:type uuu.) => (vvv rdf:type xxx.)
    for subClass, superClass in myGraph.subject_objects(RDFS.subClassOf):
        for instance, rdfType, klass in myGraph.triples((None, RDF.type, subClass)):
            inferredTriples.add((instance, RDF.type, superClass))
            print("(rdfs9)", instance, "rdf:type", superClass)

    # rdfs11: (uuu rdfs:subClassOf vvv.) and (vvv rdfs:subClassOf xxx.)
    #         => (uuu rdfs:subClassOf xxx.)
    for subClass, midClass in myGraph.subject_objects(RDFS.subClassOf):
        for midClass2, superClass in myGraph.subject_objects(RDFS.subClassOf):
            if midClass == midClass2:
                inferredTriples.add((subClass, RDFS.subClassOf, superClass))
                print("(rdfs11)", subClass, "rdfs:subClassOf", superClass)


    #I like the way you guys make this look
    print("---------------------------------")
    print("Number of inferred triples:", len(inferredTriples))
    print("---------------------------------")

myRDFSreasoner(g)  



(rdfs2) http://example.com/kad/Amsterdam rdf:type http://example.com/kad/Location
(rdfs2) http://example.com/kad/Netherlands rdf:type http://example.com/kad/Country
(rdfs3) http://example.com/kad/Germany rdf:type http://example.com/kad/Location
(rdfs3) http://example.com/kad/Rotterdam rdf:type http://example.com/kad/City
(rdfs3) http://example.com/kad/Amsterdam rdf:type http://example.com/kad/Capital
(rdfs3) http://example.com/kad/Berlin rdf:type http://example.com/kad/Capital
(rdfs5) http://example.com/kad/neighbours rdfs:subPropertyOf http://example.com/kad/Germany
(rdfs5) http://example.com/kad/hasCapital rdfs:subPropertyOf http://example.com/kad/Rotterdam
(rdfs5) http://example.com/kad/containsCity rdfs:subPropertyOf http://example.com/kad/Ijsselmeer
(rdfs7) http://example.com/kad/Netherlands http://example.com/kad/closeBy http://example.com/kad/Belgium
(rdfs7) http://example.com/kad/Netherlands http://example.com/kad/containsCity http://example.com/kad/Amsterdam
(rdfs7) http://exa

## Task 3: (20 points) Build your very own RDFS knowledge graph. 


Define a small RDF Schema vocabulary in Turtle. You can choose your own domain (e.g. movies, geography, sports), as long as it hasn't been used as an example during the lectures. The following rules must be respected:
*	The schema should define at least 4 classes, 4 properties, and 4 instances. <font color='white'>All should involve ducks.</font>
*   The properties should be used to relate the instances (i.e., object-type relations)
*	The instances should be members of at least one of the 4 defined classes
*	All resources should have an rdfs:label attribute in a suitable language.

You should use (at least) the following language features of RDF and RDFS:
* 	rdf:type (or 'a')
* 	rdfs:subClassOf
* 	rdfs:subPropertyOf
* 	rdfs:domain and rdfs:range
*	rdfs:label

Be sure to define the 'rdf:' and 'rdfs:' namespace prefixes for RDF and RDF Schema in your file (perhaps have a look at http://prefix.cc)

For creating your vocabulary you should add the axioms directly (programatically) to your Knowledge Graph as you did last week. 

Play around with the inference rules you have created in the previous task to make sure that you added some implicit knowledge, that becomes "visible" via inferencing (this will be useful for the next task). 

Finally:
- Add the knowledge you created into the RDFlib graph datastructure *myRDFSgraph*, 
- Print *myRDFSgraph* in Turtle so that we can check your "design"
- Save *myRDFSgraph* into a new file 'myRDFSgraph.ttl' (it is not necessary to submit this file)

In [ ]:
#DISCLAIMER, I like climbing and was curious 
#how much climbing stuff i could think of in 20 minutes
#sorry if you have to read through all of this
myRDFSgraph = Graph()

EX = Namespace("http://example.org/climbing/")
XSD = Namespace("http://www.w3.org/2001/XMLSchema#")

#classes
myRDFSgraph.add((EX.Hold, RDF.type, RDFS.Class))
myRDFSgraph.add((EX.Wall, RDF.type, RDFS.Class))
myRDFSgraph.add((EX.Discipline, RDF.type, RDFS.Class))
myRDFSgraph.add((EX.Equipment, RDF.type, RDFS.Class))
myRDFSgraph.add((EX.Climber, RDF.type, RDFS.Class))

#hold subclasses
myRDFSgraph.add((EX.Jug, RDFS.subClassOf, EX.Hold))
myRDFSgraph.add((EX.Crimp, RDFS.subClassOf, EX.Hold))
myRDFSgraph.add((EX.Sloper, RDFS.subClassOf, EX.Hold))

#wall subclasses
myRDFSgraph.add((EX.Overhang, RDFS.subClassOf, EX.Wall))
myRDFSgraph.add((EX.Slab, RDFS.subClassOf, EX.Wall))

#discipline subclasses
myRDFSgraph.add((EX.Trad, RDFS.subClassOf, EX.Discipline))
myRDFSgraph.add((EX.Boulder, RDFS.subClassOf, EX.Discipline))
myRDFSgraph.add((EX.Speed, RDFS.subClassOf, EX.Discipline))

#equipment subclasses
myRDFSgraph.add((EX.Harness, RDFS.subClassOf, EX.Equipment))
myRDFSgraph.add((EX.Chalk, RDFS.subClassOf, EX.Equipment))
myRDFSgraph.add((EX.Rope, RDFS.subClassOf, EX.Equipment))

#properties
myRDFSgraph.add((EX.hasHold, RDF.type, RDF.Property))
myRDFSgraph.add((EX.hasWall, RDF.type, RDF.Property))
myRDFSgraph.add((EX.hasDiscipline, RDF.type, RDF.Property))
myRDFSgraph.add((EX.hasEquipment, RDF.type, RDF.Property))
myRDFSgraph.add((EX.hasSize, RDF.type, RDF.Property))
myRDFSgraph.add((EX.isClean, RDF.type, RDF.Property))

myRDFSgraph.add((EX.hasHold, RDFS.domain, EX.Climber))
myRDFSgraph.add((EX.hasHold, RDFS.range, EX.Hold))
myRDFSgraph.add((EX.hasWall, RDFS.domain, EX.Climber))
myRDFSgraph.add((EX.hasWall, RDFS.range, EX.Wall))
myRDFSgraph.add((EX.hasDiscipline, RDFS.domain, EX.Climber))
myRDFSgraph.add((EX.hasDiscipline, RDFS.range, EX.Discipline))
myRDFSgraph.add((EX.hasEquipment, RDFS.domain, EX.Climber))
myRDFSgraph.add((EX.hasEquipment, RDFS.range, EX.Equipment))
myRDFSgraph.add((EX.hasSize, RDFS.domain, EX.Hold))
myRDFSgraph.add((EX.hasSize, RDFS.range, XSD.string))
myRDFSgraph.add((EX.isClean, RDFS.domain, EX.Hold))
myRDFSgraph.add((EX.isClean, RDFS.range, XSD.string))

#climber instances
for climber in [EX.Climber1, EX.Climber2, EX.Climber3, EX.Climber4]:
    myRDFSgraph.add((climber, RDF.type, EX.Climber))
    myRDFSgraph.add((climber, RDFS.label, Literal(str(climber).split('/')[-1])))

#hold instances
for hold, typ in [(EX.Jug1, EX.Jug), (EX.Crimp1, EX.Crimp), (EX.Sloper1, EX.Sloper)]:
    myRDFSgraph.add((hold, RDF.type, typ))
    myRDFSgraph.add((hold, RDFS.label, Literal(str(hold).split('/')[-1])))

#wall instances
for wall, typ in [(EX.Overhang1, EX.Overhang), (EX.Slab1, EX.Slab)]:
    myRDFSgraph.add((wall, RDF.type, typ))
    myRDFSgraph.add((wall, RDFS.label, Literal(str(wall).split('/')[-1])))

#discipline instances
for disc, typ in [(EX.Trad1, EX.Trad), (EX.Boulder1, EX.Boulder), (EX.Speed1, EX.Speed)]:
    myRDFSgraph.add((disc, RDF.type, typ))
    myRDFSgraph.add((disc, RDFS.label, Literal(str(disc).split('/')[-1])))

#equipment instances
for eq, typ in [(EX.Harness1, EX.Harness), (EX.Chalk1, EX.Chalk), (EX.Rope1, EX.Rope)]:
    myRDFSgraph.add((eq, RDF.type, typ))
    myRDFSgraph.add((eq, RDFS.label, Literal(str(eq).split('/')[-1])))

#example relationships of really rich climbers. I wish i had this stuff
myRDFSgraph.add((EX.Climber1, EX.hasHold, EX.Jug1))
myRDFSgraph.add((EX.Climber1, EX.hasWall, EX.Overhang1))
myRDFSgraph.add((EX.Climber1, EX.hasDiscipline, EX.Boulder1))
myRDFSgraph.add((EX.Climber1, EX.hasEquipment, EX.Chalk1))

myRDFSgraph.add((EX.Climber2, EX.hasHold, EX.Crimp1))
myRDFSgraph.add((EX.Climber2, EX.hasWall, EX.Slab1))
myRDFSgraph.add((EX.Climber2, EX.hasDiscipline, EX.Trad1))
myRDFSgraph.add((EX.Climber2, EX.hasEquipment, EX.Rope1))

myRDFSgraph.add((EX.Climber3, EX.hasHold, EX.Sloper1))
myRDFSgraph.add((EX.Climber3, EX.hasWall, EX.Overhang1))
myRDFSgraph.add((EX.Climber3, EX.hasDiscipline, EX.Speed1))
myRDFSgraph.add((EX.Climber3, EX.hasEquipment, EX.Harness1))

myRDFSgraph.add((EX.Climber4, EX.hasHold, EX.Jug1))
myRDFSgraph.add((EX.Climber4, EX.hasWall, EX.Slab1))
myRDFSgraph.add((EX.Climber4, EX.hasDiscipline, EX.Boulder1))
myRDFSgraph.add((EX.Climber4, EX.hasEquipment, EX.Chalk1))

#property values
myRDFSgraph.add((EX.Jug1, EX.hasSize, Literal("big")))
myRDFSgraph.add((EX.Crimp1, EX.hasSize, Literal("small")))
myRDFSgraph.add((EX.Sloper1, EX.isClean, Literal("clean")))
myRDFSgraph.add((EX.Jug1, EX.isClean, Literal("dirty")))
myRDFSgraph.add((EX.Climber1, EX.hasSize, Literal("big muscles")))

#all the class labels, boring
for a, b in [
    (EX.Hold, "Hold"), (EX.Wall, "Wall"), (EX.Discipline, "Discipline"),
    (EX.Equipment, "Equipment"), (EX.Climber, "Climber"),
    (EX.Jug, "Jug"), (EX.Crimp, "Crimp"), (EX.Sloper, "Sloper"),
    (EX.Overhang, "Overhang"), (EX.Slab, "Slab"),
    (EX.Trad, "Trad"), (EX.Boulder, "Boulder"), (EX.Speed, "Speed"),
    (EX.Harness, "Harness"), (EX.Chalk, "Chalk"), (EX.Rope, "Rope")
]:
    myRDFSgraph.add((a, RDFS.label, Literal(b)))

print(myRDFSgraph.serialize(format='turtle'))
print("Now let's check what we can infer from your knowledge graph...")
print("The more rules you cover, the better!")
myRDFSreasoner(myRDFSgraph)

@prefix ns1: <http://example.org/climbing/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

ns1:Climber a rdfs:Class ;
    rdfs:label "Climber" .

ns1:Discipline a rdfs:Class ;
    rdfs:label "Discipline" .

ns1:Equipment a rdfs:Class ;
    rdfs:label "Equipment" .

ns1:Hold a rdfs:Class ;
    rdfs:label "Hold" .

ns1:Wall a rdfs:Class ;
    rdfs:label "Wall" .

ns1:Climber1 a ns1:Climber ;
    rdfs:label "Climber1" ;
    ns1:hasDiscipline ns1:Boulder1 ;
    ns1:hasEquipment ns1:Chalk1 ;
    ns1:hasHold ns1:Jug1 ;
    ns1:hasWall ns1:Overhang1 .

ns1:Climber2 a ns1:Climber ;
    rdfs:label "Climber2" ;
    ns1:hasDiscipline ns1:Trad1 ;
    ns1:hasEquipment ns1:Rope1 ;
    ns1:hasHold ns1:Crimp1 ;
    ns1:hasWall ns1:Slab1 .

ns1:Climber3 a ns1:Climber ;
    rdfs:label "Climber3" ;
    ns1:hasDiscipline ns1:Speed1 ;
    ns1:hasEquipment ns1:Harness1 ;
    ns1:hasHol

## Task 4 (20 points) Compare local inferences with GraphDB results

For the remaining assignments you are going to work with a triple store called GraphDB. Follow the instructins on [the GraphDB tutorial](https://github.com/ucds-vu/knowledge-data-vu/blob/master/Tutorials/Preliminaries/tutorial-GraphDB.md) to set up a working GraphDB instance.


Fire up GraphDB and create a new repository with the ID *KnowledgeAndData* (setup -> repositories -> create new repository -> GraphDB repository). Once created, import *myRDFSgraph.ttl* in your new repository (import -> "KnowledgeAndData" -> upload RDF files). Select *The Default Graph* as target graph and press import. Wait a few seconds for the process to complete and check whether your data has been succesfully imported by using the *Explore* function.

Formulate two different SPARQL queries, and write a Python code that executes these queries over your GraphDB SPARQL endpoint (check example of Task 1).

**Each SPARQL query should return a different type of inferred knowledge** (at least one triple that was not explicitly asserted in the graph).

Specify below next to your query  (using a comment '# ...')  which type of RDFS rule is the GraphDB reasoner using to infer this answer (rdfs2, rdfs3, rdfs5, rdfs7, rdfs9, rdfs11). 

In [26]:
# Get your GraphDB repository name (setup -> repositories) and assign it to the variable 'repositoryName' below. 
repositoryName = "KnowledgeAndData"

myEndpoint = f"http://127.0.0.1:7200/repositories/{repositoryName}"
sparql = SPARQLWrapper(myEndpoint)

In [ ]:
# Query 1 - rdfs9: an instance of a subclass is also an instance of its superclass.

sparql.setQuery("""
PREFIX ex: <http://example.org/climbing/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?hold ?label
WHERE {
    ?hold rdf:type ex:Hold ;
          rdfs:label ?label .
}
ORDER BY ?hold
""")
sparql.setReturnFormat(JSON)
results = sparql.query().convert()

for row in results["results"]["bindings"]:
    print(row["hold"]["value"], row["label"]["value"])


In [28]:
# Query 2 - rdfs3: an object's type is inferred from the property's rdfs:range.

sparql.setQuery("""
PREFIX ex: <http://example.org/climbing/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

SELECT ?hold ?size
WHERE {
    ?hold ex:hasSize ?size .
    ?size rdf:type xsd:string .
}
ORDER BY ?hold
""")
sparql.setReturnFormat(JSON)
results = sparql.query().convert()

for row in results["results"]["bindings"]:
    print(row["hold"]["value"], row["size"]["value"])


## Submitting the assignment

Please submit this notebook (.ipynb) once you're finished with the assignment. It is not necessary to submit the created turtle files.